In [6]:
"""
Campaign causal analysis pipeline.

Implements, in order:
  Part 1 : estimand definition (see project-breakdown-part1.md) + episode table
  Step 3 : transaction unit / zero audit
  Outcome construction (Y columns) from the transaction file
  Plan 1 : matched stacked event-study Difference-in-Differences

Assumptions baked in from prior analysis of the files (see project-breakdown-part1.md):
  - Treatment D=1 is CAMPAIGN ASSIGNMENT (campaign_table.csv), never redemption.
  - coupon.csv must be joined on (CAMPAIGN, COUPON_UPC), and deduplicated on
    (CAMPAIGN, COUPON_UPC, PRODUCT_ID) before counting eligible products.
  - Households linked to overlapping campaigns are flagged, not silently pooled.
  - Effects are reported per campaign-week, not raw campaign totals, because
    campaign duration ranges 33-162 days.

This has NOT been run against real data yet -- it was written against the
documented schemas only. Run scripts/smoke_test in this file's __main__
block after pointing CONFIG at your actual files, and inspect intermediate
outputs before trusting anything downstream.
"""

from __future__ import annotations

import logging
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("campaign_pipeline")


# --------------------------------------------------------------------------- #
# Config
# --------------------------------------------------------------------------- #
@dataclass
class Config:
    data_dir: Path
    campaign_desc_file: str = "campaign_desc.csv"
    campaign_table_file: str = "campaign_table.csv"
    coupon_file: str = "coupon.csv"
    coupon_redempt_file: str = "coupon_redempt.csv"
    product_file: str = "product.csv"
    hh_demographic_file: str = "hh_demographic.csv"
    transaction_file: str = "transaction_data.csv"

    # Analysis parameters -- these are choices, not facts from the files.
    # State them explicitly so they're easy to challenge/change.
    pre_period_weeks: int = 4          # weeks before campaign start used as baseline
    post_period_weeks: int = 4         # weeks after campaign end used for payback check
    transaction_chunksize: int = 1_000_000  # for the ~6M row file

    paths: dict = field(init=False)

    def __post_init__(self):
        self.paths = {
            "campaign_desc": self.data_dir / self.campaign_desc_file,
            "campaign_table": self.data_dir / self.campaign_table_file,
            "coupon": self.data_dir / self.coupon_file,
            "coupon_redempt": self.data_dir / self.coupon_redempt_file,
            "product": self.data_dir / self.product_file,
            "hh_demographic": self.data_dir / self.hh_demographic_file,
            "transaction": self.data_dir / self.transaction_file,
        }


# --------------------------------------------------------------------------- #
# 1. Loading
# --------------------------------------------------------------------------- #

def load_reference_tables(cfg: Config) -> dict[str, pd.DataFrame]:
    """Load the six small reference CSVs (not the transaction file)."""
    log.info("Loading reference tables")

    campaign_desc = pd.read_csv(cfg.paths["campaign_desc"])
    campaign_table = pd.read_csv(cfg.paths["campaign_table"])
    coupon = pd.read_csv(cfg.paths["coupon"])
    coupon_redempt = pd.read_csv(cfg.paths["coupon_redempt"])
    product = pd.read_csv(cfg.paths["product"])
    hh_demographic = pd.read_csv(cfg.paths["hh_demographic"])

    # Normalize column names defensively (source files have been observed to
    # vary in case/whitespace across this dataset family).
    for df in (campaign_desc, campaign_table, coupon, coupon_redempt, product, hh_demographic):
        df.columns = [c.strip().upper() for c in df.columns]

    return {
        "campaign_desc": campaign_desc,
        "campaign_table": campaign_table,
        "coupon": coupon,
        "coupon_redempt": coupon_redempt,
        "product": product,
        "hh_demographic": hh_demographic,
    }


def dedupe_coupon_table(coupon: pd.DataFrame) -> pd.DataFrame:
    """
    coupon.csv is an eligibility table with 5,164 exact-duplicate rows (4.15%)
    across (CAMPAIGN, COUPON_UPC, PRODUCT_ID) triplets. Drop exact duplicates
    only -- do NOT collapse legitimate one-coupon-to-many-product mappings.
    """
    before = len(coupon)
    coupon_dedup = coupon.drop_duplicates(
        subset=["CAMPAIGN", "COUPON_UPC", "PRODUCT_ID"]
    ).copy()
    dropped = before - len(coupon_dedup)
    log.info(f"coupon.csv: dropped {dropped} exact-duplicate rows ({dropped/before:.2%})")
    return coupon_dedup


# --------------------------------------------------------------------------- #
# 2. Episode table (Part 1 deliverable)
# --------------------------------------------------------------------------- #

def build_episode_table(tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    Build the household x campaign episode table.
    D (treatment indicator) = row exists here = household assigned to campaign.
    Redemption is kept as a separate outcome-adjacent column, never as D.
    """
    log.info("Building episode table")

    campaign_table = tables["campaign_table"]
    campaign_desc = tables["campaign_desc"]
    coupon = dedupe_coupon_table(tables["coupon"])
    coupon_redempt = tables["coupon_redempt"]
    hh_demographic = tables["hh_demographic"]

    episodes = campaign_table.merge(
        campaign_desc[["CAMPAIGN", "DESCRIPTION", "START_DAY", "END_DAY"]],
        on="CAMPAIGN",
        how="left",
        validate="many_to_one",
    )
    episodes["DURATION_DAYS"] = episodes["END_DAY"] - episodes["START_DAY"] + 1

    # --- concurrent campaigns: count of *other* campaigns this household is
    # also linked to, whose windows overlap this campaign's window.
    episodes = _add_concurrent_campaign_count(episodes, campaign_desc)

    # --- eligible product count per campaign (post-dedup)
    eligible_counts = (
        coupon.groupby("CAMPAIGN")["PRODUCT_ID"].nunique().rename("ELIGIBLE_PRODUCT_COUNT")
    )
    episodes = episodes.merge(eligible_counts, on="CAMPAIGN", how="left")

    # --- redemption flag + first redemption day (household x campaign)
    redempt_agg = (
        coupon_redempt.groupby(["household_key".upper(), "CAMPAIGN"])["DAY"]
        .min()
        .rename("FIRST_REDEMPTION_DAY")
        .reset_index()
    )
    episodes = episodes.merge(redempt_agg, on=["HOUSEHOLD_KEY", "CAMPAIGN"], how="left")
    episodes["REDEEMED"] = episodes["FIRST_REDEMPTION_DAY"].notna().astype(int)

    # --- demographics (kept as explicit categories; missing stays missing)
    episodes = episodes.merge(hh_demographic, on="HOUSEHOLD_KEY", how="left")
    episodes["HAS_DEMOGRAPHICS"] = episodes["AGE_DESC"].notna().astype(int) if "AGE_DESC" in episodes else np.nan

    log.info(f"Episode table: {len(episodes)} rows, {episodes['HOUSEHOLD_KEY'].nunique()} households")
    return episodes


def _add_concurrent_campaign_count(episodes: pd.DataFrame, campaign_desc: pd.DataFrame) -> pd.DataFrame:
    """
    For every episode (household, campaign), count how many OTHER campaigns
    that same household is linked to whose [START_DAY, END_DAY] overlaps this
    campaign's window. O(n_households * campaigns_per_household^2) via groupby
    -- fine at 7,208 rows / max 17 campaigns per household.
    """
    windows = campaign_desc.set_index("CAMPAIGN")[["START_DAY", "END_DAY"]]

    def _count_for_group(group: pd.DataFrame) -> pd.Series:
        camps = group["CAMPAIGN"].tolist()
        counts = []
        for c in camps:
            s0, e0 = windows.loc[c, "START_DAY"], windows.loc[c, "END_DAY"]
            n_overlap = 0
            for other in camps:
                if other == c:
                    continue
                s1, e1 = windows.loc[other, "START_DAY"], windows.loc[other, "END_DAY"]
                if s0 <= e1 and s1 <= e0:
                    n_overlap += 1
            counts.append(n_overlap)
        return pd.Series(counts, index=group.index)

    episodes = episodes.copy()
    episodes["N_CONCURRENT_CAMPAIGNS"] = (
        episodes.groupby("HOUSEHOLD_KEY", group_keys=False).apply(_count_for_group)
    )
    return episodes


# --------------------------------------------------------------------------- #
# 3. Transaction audit (Step 3)
# --------------------------------------------------------------------------- #

def audit_transactions(cfg: Config) -> dict:
    """
    Streams the transaction file in chunks (it's ~6M rows) and reports:
      - rows/units carried by extreme-quantity outliers (possible fuel mixing)
      - household-week combinations with zero shopping trips
      - basic negative/zero-value integrity checks
    Does not modify the file; returns a summary dict to inform filtering
    decisions made explicitly later (never silently).
    """
    log.info("Auditing transaction file (chunked)")

    qty_values = []
    total_rows = 0
    total_qty = 0.0
    neg_sales = 0
    neg_qty = 0
    hh_week_pairs = set()
    all_households = set()
    all_weeks = set()

    for chunk in pd.read_csv(cfg.paths["transaction"], chunksize=cfg.transaction_chunksize):
        chunk.columns = [c.strip() for c in chunk.columns]
        total_rows += len(chunk)
        total_qty += chunk["QUANTITY"].sum()
        neg_sales += (chunk["SALES_VALUE"] < 0).sum()
        neg_qty += (chunk["QUANTITY"] < 0).sum()

        qty_values.append(chunk["QUANTITY"].to_numpy())

        pairs = set(zip(chunk["household_key"], chunk["WEEK_NO"]))
        hh_week_pairs |= pairs
        all_households |= set(chunk["household_key"].unique())
        all_weeks |= set(chunk["WEEK_NO"].unique())

    qty_all = np.concatenate(qty_values)
    q99 = np.quantile(qty_all, 0.99)
    outlier_mask = qty_all >= q99
    outlier_row_share = outlier_mask.mean()
    outlier_unit_share = qty_all[outlier_mask].sum() / qty_all.sum()

    n_possible_hh_weeks = len(all_households) * len(all_weeks)
    no_trip_share = 1 - (len(hh_week_pairs) / n_possible_hh_weeks) if n_possible_hh_weeks else np.nan

    summary = {
        "total_rows": total_rows,
        "total_quantity": float(total_qty),
        "negative_sales_rows": int(neg_sales),
        "negative_quantity_rows": int(neg_qty),
        "top_1pct_qty_row_share": float(outlier_row_share),
        "top_1pct_qty_unit_share": float(outlier_unit_share),
        "n_households": len(all_households),
        "n_weeks": len(all_weeks),
        "household_week_no_trip_share": float(no_trip_share),
    }
    log.info(f"Audit summary: {summary}")
    log.warning(
        "top_1pct_qty_unit_share above should be compared against the PDF's claimed "
        "98.7%-units-in-1.2%-of-rows fuel-mixing figure. If far lower, this dataset's "
        "fuel contamination may differ from the PDF's reference dataset -- do not assume "
        "it transfers."
    )
    return summary


# --------------------------------------------------------------------------- #
# 4. Outcome construction (Y columns for the episode table)
# --------------------------------------------------------------------------- #

def compute_outcomes(
    cfg: Config,
    episodes: pd.DataFrame,
    coupon: pd.DataFrame,
    product: pd.DataFrame,
) -> pd.DataFrame:
    """
    Streams the transaction file once and, per (household, campaign) episode,
    accumulates:
      Y_ELIGIBLE_SALES / Y_ELIGIBLE_UNITS : eligible-product sales during window
      Y_CATEGORY_SALES                    : same-commodity sales during window
      Y_RIVAL_SALES                       : same-commodity, NON-eligible sales during window
      Y_PRE_SALES                         : household total sales in pre_period_weeks before start
      Y_POST_SALES                        : household total sales in post_period_weeks after end

    This is a single streaming pass with running accumulators keyed by
    (household_key, campaign) -- feasible at 6M rows without loading
    everything into memory at once, but each chunk is compared against the
    full episode/eligibility tables, so keep those indexed.
    """
    log.info("Computing outcomes from transaction stream")

    eligible = coupon[["CAMPAIGN", "PRODUCT_ID"]].drop_duplicates()
    eligible_set_by_campaign = eligible.groupby("CAMPAIGN")["PRODUCT_ID"].apply(set).to_dict()

    product_commodity = product.set_index("PRODUCT_ID")["COMMODITY_DESC"]

    ep = episodes.set_index(["HOUSEHOLD_KEY", "CAMPAIGN"])
    accum = {col: {} for col in [
        "Y_ELIGIBLE_SALES", "Y_ELIGIBLE_UNITS", "Y_CATEGORY_SALES",
        "Y_RIVAL_SALES", "Y_PRE_SALES", "Y_POST_SALES",
    ]}

    # Pre-index episode windows per household for fast lookup per chunk.
    hh_windows = episodes.groupby("HOUSEHOLD_KEY").apply(
        lambda g: list(zip(g["CAMPAIGN"], g["START_DAY"], g["END_DAY"]))
    ).to_dict()

    for chunk in pd.read_csv(cfg.paths["transaction"], chunksize=cfg.transaction_chunksize):
        chunk.columns = [c.strip() for c in chunk.columns]
        chunk["COMMODITY_DESC"] = chunk["PRODUCT_ID"].map(product_commodity)

        for hh, sub in chunk.groupby("household_key"):
            windows = hh_windows.get(hh)
            if not windows:
                continue
            for campaign, start, end in windows:
                elig_products = eligible_set_by_campaign.get(campaign, set())
                pre_start = start - cfg.pre_period_weeks * 7
                post_end = end + cfg.post_period_weeks * 7

                during = sub[(sub["DAY"] >= start) & (sub["DAY"] <= end)]
                pre = sub[(sub["DAY"] >= pre_start) & (sub["DAY"] < start)]
                post = sub[(sub["DAY"] > end) & (sub["DAY"] <= post_end)]

                if len(during):
                    is_elig = during["PRODUCT_ID"].isin(elig_products)
                    same_commodity = during["COMMODITY_DESC"].notna()  # placeholder join key check
                    key = (hh, campaign)
                    accum["Y_ELIGIBLE_SALES"][key] = accum["Y_ELIGIBLE_SALES"].get(key, 0) + during.loc[is_elig, "SALES_VALUE"].sum()
                    accum["Y_ELIGIBLE_UNITS"][key] = accum["Y_ELIGIBLE_UNITS"].get(key, 0) + during.loc[is_elig, "QUANTITY"].sum()
                    accum["Y_CATEGORY_SALES"][key] = accum["Y_CATEGORY_SALES"].get(key, 0) + during["SALES_VALUE"].sum()
                    accum["Y_RIVAL_SALES"][key] = accum["Y_RIVAL_SALES"].get(key, 0) + during.loc[~is_elig, "SALES_VALUE"].sum()
                if len(pre):
                    key = (hh, campaign)
                    accum["Y_PRE_SALES"][key] = accum["Y_PRE_SALES"].get(key, 0) + pre["SALES_VALUE"].sum()
                if len(post):
                    key = (hh, campaign)
                    accum["Y_POST_SALES"][key] = accum["Y_POST_SALES"].get(key, 0) + post["SALES_VALUE"].sum()

    for col, d in accum.items():
        idx = pd.MultiIndex.from_tuples(d.keys(), names=["HOUSEHOLD_KEY", "CAMPAIGN"])
        s = pd.Series(d.values(), index=idx, name=col)
        ep = ep.join(s, how="left")

    episodes_out = ep.reset_index()
    for col in accum:
        episodes_out[col] = episodes_out[col].fillna(0.0)

    log.warning(
        "Y_CATEGORY_SALES currently uses COMMODITY_DESC as the category key but does not "
        "restrict to the SAME commodity as the eligible products for that campaign -- fix "
        "before trusting rival/category numbers: join eligible products to their own "
        "COMMODITY_DESC first, then filter transactions to that same commodity set."
    )
    return episodes_out


# --------------------------------------------------------------------------- #
# 5. Plan 1: matched stacked event-study DiD
# --------------------------------------------------------------------------- #

def run_plan1_event_study_did(episodes: pd.DataFrame, outcome_col: str = "Y_ELIGIBLE_SALES") -> pd.DataFrame:
    """
    For each campaign:
      1. Treated = households assigned to this campaign (N_CONCURRENT_CAMPAIGNS == 0
         by default -- concurrently-treated households are excluded from Plan 1,
         not modeled, since Plan 1 can't separate simultaneous treatments).
      2. Controls = households NOT assigned to this campaign, and not themselves
         under any other campaign whose window overlaps this one.
      3. Match treated to controls on available demographics + pre-period sales
         (nearest-neighbor on a simple propensity score from logistic regression).
      4. DiD estimate = (during_treated - pre_treated) - (during_control - pre_control),
         normalized to per-week.
    Returns one row per campaign with the DiD estimate and a naive SE from the
    matched-pair variance (a placeholder -- the PDF wants bootstrap/Fieller-style
    intervals for the ROI stage, not this SE, but this is enough to rank
    campaigns directionally).
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.neighbors import NearestNeighbors

    log.info("Running Plan 1: matched stacked event-study DiD")
    results = []

    demo_cols = [c for c in episodes.columns if c.endswith("_DESC")]
    feature_cols = demo_cols + ["Y_PRE_SALES"]

    for campaign, camp_df in episodes.groupby("CAMPAIGN"):
        duration_weeks = camp_df["DURATION_DAYS"].iloc[0] / 7.0

        eligible_pool = episodes[episodes["N_CONCURRENT_CAMPAIGNS"].eq(0)]
        treated = eligible_pool[
            (eligible_pool["CAMPAIGN"] == campaign)
        ].copy()
        controls_pool = eligible_pool[
            (eligible_pool["CAMPAIGN"] != campaign)
        ].drop_duplicates(subset="HOUSEHOLD_KEY").copy()
        # Exclude any household that IS linked to this campaign, from the control pool
        controls_pool = controls_pool[
            ~controls_pool["HOUSEHOLD_KEY"].isin(treated["HOUSEHOLD_KEY"])
        ]

        if len(treated) < 10 or len(controls_pool) < 10:
            log.info(f"Campaign {campaign}: too few treated/control households after exclusions, skipping direct estimate (needs pooling)")
            results.append({"CAMPAIGN": campaign, "N_TREATED": len(treated), "N_CONTROL": len(controls_pool), "DID_ESTIMATE_PER_WEEK": np.nan, "NOTE": "insufficient sample -- requires hierarchical pooling"})
            continue

        combo = pd.concat([treated.assign(_T=1), controls_pool.assign(_T=0)], ignore_index=True)
        X = pd.get_dummies(combo[feature_cols], dummy_na=True)
        X = X.fillna(0)

        try:
            ps_model = LogisticRegression(max_iter=1000)
            ps_model.fit(X, combo["_T"])
            combo["_pscore"] = ps_model.predict_proba(X)[:, 1]
        except Exception as e:
            log.warning(f"Campaign {campaign}: propensity model failed ({e}), using pre-period sales only for matching")
            combo["_pscore"] = combo["Y_PRE_SALES"]

        treated_idx = combo[combo["_T"] == 1].index
        control_idx = combo[combo["_T"] == 0].index
        nn = NearestNeighbors(n_neighbors=1).fit(combo.loc[control_idx, ["_pscore"]])
        _, match_pos = nn.kneighbors(combo.loc[treated_idx, ["_pscore"]])
        matched_control_idx = combo.loc[control_idx].iloc[match_pos.flatten()].index

        treated_during = combo.loc[treated_idx, outcome_col].to_numpy()
        treated_pre = combo.loc[treated_idx, "Y_PRE_SALES"].to_numpy()
        control_during = combo.loc[matched_control_idx, outcome_col].to_numpy()
        control_pre = combo.loc[matched_control_idx, "Y_PRE_SALES"].to_numpy()

        # NOTE: control households' "during" outcome should be measured over the
        # SAME calendar window as the treated campaign, which Y_ELIGIBLE_SALES
        # already is per-episode for treated rows -- for controls (not linked to
        # this campaign) you must separately pull their spend in this campaign's
        # window, not their own episode's window. This placeholder reuses their
        # own episode outcome as an approximation and MUST be replaced with a
        # direct transaction-window pull before trusting these numbers.
        did_per_pair = (treated_during - treated_pre) - (control_during - control_pre)
        did_estimate = np.nanmean(did_per_pair) / max(duration_weeks, 1e-9)
        se = np.nanstd(did_per_pair, ddof=1) / np.sqrt(len(did_per_pair)) / max(duration_weeks, 1e-9)

        results.append({
            "CAMPAIGN": campaign,
            "N_TREATED": len(treated),
            "N_CONTROL": len(controls_pool),
            "DID_ESTIMATE_PER_WEEK": did_estimate,
            "SE_PER_WEEK": se,
            "NOTE": "matched, single campaign -- pool across campaigns before reporting",
        })

    return pd.DataFrame(results)


# --------------------------------------------------------------------------- #
# 6. Orchestration
# --------------------------------------------------------------------------- #

def run_pipeline(cfg: Config) -> dict:
    tables = load_reference_tables(cfg)
    episodes = build_episode_table(tables)
    audit_summary = audit_transactions(cfg)
    episodes_with_outcomes = compute_outcomes(cfg, episodes, dedupe_coupon_table(tables["coupon"]), tables["product"])
    plan1_results = run_plan1_event_study_did(episodes_with_outcomes)

    return {
        "episodes": episodes_with_outcomes,
        "audit_summary": audit_summary,
        "plan1_results": plan1_results,
    }


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", type=str, required=True, help="Directory containing all 7 CSVs")
    parser.add_argument("--out-dir", type=str, default="./pipeline_output")
    args = parser.parse_args(["--data-dir", "data"])    
    cfg = Config(data_dir=Path(args.data_dir))
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    outputs = run_pipeline(cfg)
    outputs["episodes"].to_csv(out_dir / "episode_table_with_outcomes.csv", index=False)
    outputs["plan1_results"].to_csv(out_dir / "plan1_did_results.csv", index=False)
    log.info(f"Done. Outputs written to {out_dir}")

2026-08-06 11:59:30,882 | INFO | Loading reference tables
2026-08-06 11:59:30,986 | INFO | Building episode table
2026-08-06 11:59:30,993 | INFO | coupon.csv: dropped 5164 exact-duplicate rows (4.15%)
2026-08-06 11:59:31,859 | INFO | Episode table: 7208 rows, 1584 households
2026-08-06 11:59:31,859 | INFO | Auditing transaction file (chunked)
2026-08-06 11:59:33,206 | INFO | Audit summary: {'total_rows': 2595732, 'total_quantity': 260685622.0, 'negative_sales_rows': 0, 'negative_quantity_rows': 0, 'top_1pct_qty_row_share': 0.010642470023869952, 'top_1pct_qty_unit_share': 0.9873510054958076, 'n_households': 2500, 'n_weeks': 102, 'household_week_no_trip_share': 0.5138196078431372}
2026-08-06 11:59:33,206 | WARNING | top_1pct_qty_unit_share above should be compared against the PDF's claimed 98.7%-units-in-1.2%-of-rows fuel-mixing figure. If far lower, this dataset's fuel contamination may differ from the PDF's reference dataset -- do not assume it transfers.
2026-08-06 11:59:33,221 | INFO

In [11]:
"""
Campaign causal analysis pipeline.

Implements, in order:
  Part 1 : estimand definition (see project-breakdown-part1.md) + episode table
  Step 3 : transaction unit / zero audit
  Outcome construction (Y columns) from the transaction file
  Plan 1 : matched stacked event-study Difference-in-Differences

Assumptions baked in from prior analysis of the files (see project-breakdown-part1.md):
  - Treatment D=1 is CAMPAIGN ASSIGNMENT (campaign_table.csv), never redemption.
  - coupon.csv must be joined on (CAMPAIGN, COUPON_UPC), and deduplicated on
    (CAMPAIGN, COUPON_UPC, PRODUCT_ID) before counting eligible products.
  - Households linked to overlapping campaigns are flagged, not silently pooled.
  - Effects are reported per campaign-week, not raw campaign totals, because
    campaign duration ranges 33-162 days.

This has NOT been run against real data yet -- it was written against the
documented schemas only. Run scripts/smoke_test in this file's __main__
block after pointing CONFIG at your actual files, and inspect intermediate
outputs before trusting anything downstream.
"""

from __future__ import annotations

import logging
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("campaign_pipeline")


# --------------------------------------------------------------------------- #
# Config
# --------------------------------------------------------------------------- #

@dataclass
class Config:
    data_dir: Path
    campaign_desc_file: str = "07-campaign_desc.csv"
    campaign_table_file: str = "02-campaign_table.csv"
    coupon_file: str = "05-coupon.csv"
    coupon_redempt_file: str = "01-coupon_redempt.csv"
    product_file: str = "04-product.csv"
    hh_demographic_file: str = "03-hh_demographic.csv"
    transaction_file: str = "transaction_data.csv"

    # Analysis parameters -- these are choices, not facts from the files.
    # State them explicitly so they're easy to challenge/change.
    pre_period_weeks: int = 4          # weeks before campaign start used as baseline
    post_period_weeks: int = 4         # weeks after campaign end used for payback check
    transaction_chunksize: int = 1_000_000  # for the ~6M row file

    paths: dict = field(init=False)

    def __post_init__(self):
        self.paths = {
            "campaign_desc": self.data_dir / self.campaign_desc_file,
            "campaign_table": self.data_dir / self.campaign_table_file,
            "coupon": self.data_dir / self.coupon_file,
            "coupon_redempt": self.data_dir / self.coupon_redempt_file,
            "product": self.data_dir / self.product_file,
            "hh_demographic": self.data_dir / self.hh_demographic_file,
            "transaction": self.data_dir / self.transaction_file,
        }


# --------------------------------------------------------------------------- #
# 1. Loading
# --------------------------------------------------------------------------- #

def load_reference_tables(cfg: Config) -> dict[str, pd.DataFrame]:
    """Load the six small reference CSVs (not the transaction file)."""
    log.info("Loading reference tables")

    campaign_desc = pd.read_csv(cfg.paths["campaign_desc"])
    campaign_table = pd.read_csv(cfg.paths["campaign_table"])
    coupon = pd.read_csv(cfg.paths["coupon"])
    coupon_redempt = pd.read_csv(cfg.paths["coupon_redempt"])
    product = pd.read_csv(cfg.paths["product"])
    hh_demographic = pd.read_csv(cfg.paths["hh_demographic"])

    # Normalize column names defensively (source files have been observed to
    # vary in case/whitespace across this dataset family).
    for df in (campaign_desc, campaign_table, coupon, coupon_redempt, product, hh_demographic):
        df.columns = [c.strip().upper() for c in df.columns]

    return {
        "campaign_desc": campaign_desc,
        "campaign_table": campaign_table,
        "coupon": coupon,
        "coupon_redempt": coupon_redempt,
        "product": product,
        "hh_demographic": hh_demographic,
    }


def dedupe_coupon_table(coupon: pd.DataFrame) -> pd.DataFrame:
    """
    coupon.csv is an eligibility table with 5,164 exact-duplicate rows (4.15%)
    across (CAMPAIGN, COUPON_UPC, PRODUCT_ID) triplets. Drop exact duplicates
    only -- do NOT collapse legitimate one-coupon-to-many-product mappings.
    """
    before = len(coupon)
    coupon_dedup = coupon.drop_duplicates(
        subset=["CAMPAIGN", "COUPON_UPC", "PRODUCT_ID"]
    ).copy()
    dropped = before - len(coupon_dedup)
    log.info(f"coupon.csv: dropped {dropped} exact-duplicate rows ({dropped/before:.2%})")
    return coupon_dedup


# --------------------------------------------------------------------------- #
# 2. Episode table (Part 1 deliverable)
# --------------------------------------------------------------------------- #

def build_episode_table(tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    Build the household x campaign episode table.
    D (treatment indicator) = row exists here = household assigned to campaign.
    Redemption is kept as a separate outcome-adjacent column, never as D.
    """
    log.info("Building episode table")

    campaign_table = tables["campaign_table"]
    campaign_desc = tables["campaign_desc"]
    coupon = dedupe_coupon_table(tables["coupon"])
    coupon_redempt = tables["coupon_redempt"]
    hh_demographic = tables["hh_demographic"]

    episodes = campaign_table.merge(
        campaign_desc[["CAMPAIGN", "DESCRIPTION", "START_DAY", "END_DAY"]],
        on="CAMPAIGN",
        how="left",
        validate="many_to_one",
    )
    episodes["DURATION_DAYS"] = episodes["END_DAY"] - episodes["START_DAY"] + 1

    # --- concurrent campaigns: count of *other* campaigns this household is
    # also linked to, whose windows overlap this campaign's window.
    episodes = _add_concurrent_campaign_count(episodes, campaign_desc)

    # --- eligible product count per campaign (post-dedup)
    eligible_counts = (
        coupon.groupby("CAMPAIGN")["PRODUCT_ID"].nunique().rename("ELIGIBLE_PRODUCT_COUNT")
    )
    episodes = episodes.merge(eligible_counts, on="CAMPAIGN", how="left")

    # --- redemption flag + first redemption day (household x campaign)
    redempt_agg = (
        coupon_redempt.groupby(["household_key".upper(), "CAMPAIGN"])["DAY"]
        .min()
        .rename("FIRST_REDEMPTION_DAY")
        .reset_index()
    )
    episodes = episodes.merge(redempt_agg, on=["HOUSEHOLD_KEY", "CAMPAIGN"], how="left")
    episodes["REDEEMED"] = episodes["FIRST_REDEMPTION_DAY"].notna().astype(int)

    # --- demographics (kept as explicit categories; missing stays missing)
    episodes = episodes.merge(hh_demographic, on="HOUSEHOLD_KEY", how="left")
    episodes["HAS_DEMOGRAPHICS"] = episodes["AGE_DESC"].notna().astype(int) if "AGE_DESC" in episodes else np.nan

    log.info(f"Episode table: {len(episodes)} rows, {episodes['HOUSEHOLD_KEY'].nunique()} households")
    return episodes


def _add_concurrent_campaign_count(episodes: pd.DataFrame, campaign_desc: pd.DataFrame) -> pd.DataFrame:
    """
    For every episode (household, campaign), count how many OTHER campaigns
    that same household is linked to whose [START_DAY, END_DAY] overlaps this
    campaign's window. O(n_households * campaigns_per_household^2) via groupby
    -- fine at 7,208 rows / max 17 campaigns per household.
    """
    windows = campaign_desc.set_index("CAMPAIGN")[["START_DAY", "END_DAY"]]

    def _count_for_group(group: pd.DataFrame) -> pd.Series:
        camps = group["CAMPAIGN"].tolist()
        counts = []
        for c in camps:
            s0, e0 = windows.loc[c, "START_DAY"], windows.loc[c, "END_DAY"]
            n_overlap = 0
            for other in camps:
                if other == c:
                    continue
                s1, e1 = windows.loc[other, "START_DAY"], windows.loc[other, "END_DAY"]
                if s0 <= e1 and s1 <= e0:
                    n_overlap += 1
            counts.append(n_overlap)
        return pd.Series(counts, index=group.index)

    episodes = episodes.copy()
    episodes["N_CONCURRENT_CAMPAIGNS"] = (
        episodes.groupby("HOUSEHOLD_KEY", group_keys=False).apply(_count_for_group)
    )
    return episodes


# --------------------------------------------------------------------------- #
# 3. Transaction audit (Step 3)
# --------------------------------------------------------------------------- #

def audit_transactions(cfg: Config) -> dict:
    """
    Streams the transaction file in chunks (it's ~6M rows) and reports:
      - rows/units carried by extreme-quantity outliers (possible fuel mixing)
      - household-week combinations with zero shopping trips
      - basic negative/zero-value integrity checks
    Does not modify the file; returns a summary dict to inform filtering
    decisions made explicitly later (never silently).
    """
    log.info("Auditing transaction file (chunked)")

    qty_values = []
    total_rows = 0
    total_qty = 0.0
    neg_sales = 0
    neg_qty = 0
    hh_week_pairs = set()
    all_households = set()
    all_weeks = set()

    for chunk in pd.read_csv(cfg.paths["transaction"], chunksize=cfg.transaction_chunksize):
        chunk.columns = [c.strip() for c in chunk.columns]
        total_rows += len(chunk)
        total_qty += chunk["QUANTITY"].sum()
        neg_sales += (chunk["SALES_VALUE"] < 0).sum()
        neg_qty += (chunk["QUANTITY"] < 0).sum()

        qty_values.append(chunk["QUANTITY"].to_numpy())

        pairs = set(zip(chunk["household_key"], chunk["WEEK_NO"]))
        hh_week_pairs |= pairs
        all_households |= set(chunk["household_key"].unique())
        all_weeks |= set(chunk["WEEK_NO"].unique())

    qty_all = np.concatenate(qty_values)
    q99 = np.quantile(qty_all, 0.99)
    outlier_mask = qty_all >= q99
    outlier_row_share = outlier_mask.mean()
    outlier_unit_share = qty_all[outlier_mask].sum() / qty_all.sum()

    n_possible_hh_weeks = len(all_households) * len(all_weeks)
    no_trip_share = 1 - (len(hh_week_pairs) / n_possible_hh_weeks) if n_possible_hh_weeks else np.nan

    summary = {
        "total_rows": total_rows,
        "total_quantity": float(total_qty),
        "negative_sales_rows": int(neg_sales),
        "negative_quantity_rows": int(neg_qty),
        "top_1pct_qty_row_share": float(outlier_row_share),
        "top_1pct_qty_unit_share": float(outlier_unit_share),
        "n_households": len(all_households),
        "n_weeks": len(all_weeks),
        "household_week_no_trip_share": float(no_trip_share),
    }
    log.info(f"Audit summary: {summary}")
    log.warning(
        "top_1pct_qty_unit_share above should be compared against the PDF's claimed "
        "98.7%-units-in-1.2%-of-rows fuel-mixing figure. If far lower, this dataset's "
        "fuel contamination may differ from the PDF's reference dataset -- do not assume "
        "it transfers."
    )
    return summary


# --------------------------------------------------------------------------- #
# 4. Outcome construction (Y columns for the episode table)
# --------------------------------------------------------------------------- #

def compute_household_campaign_outcomes(
    cfg: Config,
    campaign_desc: pd.DataFrame,
    coupon: pd.DataFrame,
    product: pd.DataFrame,
    households_in_scope: set,
) -> pd.DataFrame:
    """
    Streams the transaction file ONCE and computes, for EVERY household in
    households_in_scope x EVERY campaign (not just linked pairs), outcomes
    over that campaign's calendar window:
      Y_ELIGIBLE_SALES / Y_ELIGIBLE_UNITS : sales of THAT campaign's eligible products
      Y_CATEGORY_SALES                    : sales in the SAME COMMODITY as that
                                             campaign's eligible products (not all sales)
      Y_RIVAL_SALES                       : Y_CATEGORY_SALES minus Y_ELIGIBLE_SALES
                                             (same commodity, non-eligible products)
      Y_PRE_SALES / Y_POST_SALES          : household total sales in the pre/post windows

    Computing this for every household (linked or not) against every campaign
    is what makes Plan 1's control group comparable: a control household's
    "during" outcome is now measured over the SAME calendar window as the
    campaign it's being matched against, not over its own unrelated episode.

    30 campaigns x ~2,500 households is a small enough combination to hold in
    memory; only the streaming transaction read needs chunking.
    """
    log.info("Computing household x campaign window outcomes (universal table)")

    # Eligible products per campaign, and the commodities those products belong to.
    eligible = coupon[["CAMPAIGN", "PRODUCT_ID"]].drop_duplicates()
    product_commodity = product.set_index("PRODUCT_ID")["COMMODITY_DESC"]
    eligible = eligible.assign(COMMODITY_DESC=eligible["PRODUCT_ID"].map(product_commodity))

    eligible_products_by_campaign = eligible.groupby("CAMPAIGN")["PRODUCT_ID"].apply(set).to_dict()
    eligible_commodities_by_campaign = (
        eligible.groupby("CAMPAIGN")["COMMODITY_DESC"].apply(lambda s: set(s.dropna())).to_dict()
    )

    windows = campaign_desc.set_index("CAMPAIGN")[["START_DAY", "END_DAY"]]
    campaigns = windows.index.tolist()

    accum = {col: {} for col in [
        "Y_ELIGIBLE_SALES", "Y_ELIGIBLE_UNITS", "Y_CATEGORY_SALES",
        "Y_RIVAL_SALES", "Y_PRE_SALES", "Y_POST_SALES",
    ]}

    for chunk in pd.read_csv(cfg.paths["transaction"], chunksize=cfg.transaction_chunksize):
        chunk.columns = [c.strip() for c in chunk.columns]
        chunk = chunk[chunk["household_key"].isin(households_in_scope)]
        if chunk.empty:
            continue
        chunk["COMMODITY_DESC"] = chunk["PRODUCT_ID"].map(product_commodity)

        for hh, sub in chunk.groupby("household_key"):
            for campaign in campaigns:
                start, end = windows.loc[campaign, "START_DAY"], windows.loc[campaign, "END_DAY"]
                pre_start = start - cfg.pre_period_weeks * 7
                post_end = end + cfg.post_period_weeks * 7

                during = sub[(sub["DAY"] >= start) & (sub["DAY"] <= end)]
                pre = sub[(sub["DAY"] >= pre_start) & (sub["DAY"] < start)]
                post = sub[(sub["DAY"] > end) & (sub["DAY"] <= post_end)]

                key = (hh, campaign)
                if len(during):
                    elig_products = eligible_products_by_campaign.get(campaign, set())
                    elig_commodities = eligible_commodities_by_campaign.get(campaign, set())
                    is_elig = during["PRODUCT_ID"].isin(elig_products)
                    same_commodity = during["COMMODITY_DESC"].isin(elig_commodities)

                    elig_sales = during.loc[is_elig, "SALES_VALUE"].sum()
                    category_sales = during.loc[same_commodity, "SALES_VALUE"].sum()

                    accum["Y_ELIGIBLE_SALES"][key] = accum["Y_ELIGIBLE_SALES"].get(key, 0) + elig_sales
                    accum["Y_ELIGIBLE_UNITS"][key] = accum["Y_ELIGIBLE_UNITS"].get(key, 0) + during.loc[is_elig, "QUANTITY"].sum()
                    accum["Y_CATEGORY_SALES"][key] = accum["Y_CATEGORY_SALES"].get(key, 0) + category_sales
                    accum["Y_RIVAL_SALES"][key] = accum["Y_RIVAL_SALES"].get(key, 0) + (category_sales - elig_sales)
                if len(pre):
                    accum["Y_PRE_SALES"][key] = accum["Y_PRE_SALES"].get(key, 0) + pre["SALES_VALUE"].sum()
                if len(post):
                    accum["Y_POST_SALES"][key] = accum["Y_POST_SALES"].get(key, 0) + post["SALES_VALUE"].sum()

    all_keys = set()
    for d in accum.values():
        all_keys |= set(d.keys())
    idx = pd.MultiIndex.from_tuples(sorted(all_keys), names=["HOUSEHOLD_KEY", "CAMPAIGN"])
    out = pd.DataFrame(index=idx)
    for col, d in accum.items():
        out[col] = pd.Series(d)
    out = out.fillna(0.0).reset_index()

    log.info(f"Universal outcome table: {len(out)} household x campaign rows for {len(households_in_scope)} households")
    return out


# --------------------------------------------------------------------------- #
# 5. Plan 1: matched stacked event-study DiD
# --------------------------------------------------------------------------- #

def run_plan1_event_study_did(
    episodes: pd.DataFrame,
    universal_outcomes: pd.DataFrame,
    demographics: pd.DataFrame,
    outcome_col: str = "Y_ELIGIBLE_SALES",
) -> pd.DataFrame:
    """
    For each campaign:
      1. Treated = households assigned to this campaign with N_CONCURRENT_CAMPAIGNS == 0
         (concurrently-treated households are excluded from Plan 1, not modeled --
         Plan 1 can't separate simultaneous treatments).
      2. Controls = every OTHER household in universal_outcomes that is NOT linked
         to this campaign in episodes, regardless of what campaigns they're linked
         to elsewhere -- their outcome is pulled from universal_outcomes for THIS
         campaign's calendar window, so treated and control are compared over the
         identical time period. (Controls linked to a campaign that overlaps this
         one's window are still excluded, since their own treatment would confound
         the comparison.)
      3. Match treated to controls on available demographics + pre-period sales
         (nearest-neighbor on a propensity score from logistic regression).
      4. DiD estimate = (during_treated - pre_treated) - (during_control - pre_control),
         normalized to per-week.
    Returns one row per campaign with the DiD estimate and a naive matched-pair SE
    (a placeholder -- the PDF wants bootstrap/Fieller-style intervals at the ROI
    stage, not this SE; this is enough to rank campaigns directionally, not to
    report as a final interval).
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.neighbors import NearestNeighbors

    log.info("Running Plan 1: matched stacked event-study DiD")
    results = []

    # household -> set of campaigns they're linked to, for exclusion checks
    linked_campaigns_by_hh = episodes.groupby("HOUSEHOLD_KEY")["CAMPAIGN"].apply(set).to_dict()
    concurrent_free_hh = set(episodes.loc[episodes["N_CONCURRENT_CAMPAIGNS"].eq(0), "HOUSEHOLD_KEY"])

    windows = episodes.drop_duplicates("CAMPAIGN").set_index("CAMPAIGN")[["START_DAY", "END_DAY", "DURATION_DAYS"]]
    all_campaign_windows = windows[["START_DAY", "END_DAY"]]

    demo_cols = [c for c in demographics.columns if c.endswith("_DESC")]
    uo = universal_outcomes.merge(demographics, on="HOUSEHOLD_KEY", how="left")

    for campaign in windows.index:
        c_start, c_end = windows.loc[campaign, ["START_DAY", "END_DAY"]]
        duration_weeks = windows.loc[campaign, "DURATION_DAYS"] / 7.0

        treated_hh = set(episodes.loc[episodes["CAMPAIGN"] == campaign, "HOUSEHOLD_KEY"]) & concurrent_free_hh

        def _overlaps_campaign(other_campaign: int) -> bool:
            o_start, o_end = all_campaign_windows.loc[other_campaign]
            return c_start <= o_end and o_start <= c_end

        overlapping_campaigns = {c for c in all_campaign_windows.index if c != campaign and _overlaps_campaign(c)}

        control_hh = {
            hh for hh, camps in linked_campaigns_by_hh.items()
            if hh not in treated_hh and not (camps & ({campaign} | overlapping_campaigns))
        }
        # Households present in the transaction data but never linked to any campaign are valid controls too.
        control_hh |= set(universal_outcomes["HOUSEHOLD_KEY"].unique()) - set(linked_campaigns_by_hh.keys()) - treated_hh

        treated_rows = uo[(uo["CAMPAIGN"] == campaign) & (uo["HOUSEHOLD_KEY"].isin(treated_hh))].copy()
        control_rows = uo[(uo["CAMPAIGN"] == campaign) & (uo["HOUSEHOLD_KEY"].isin(control_hh))].copy()

        if len(treated_rows) < 10 or len(control_rows) < 10:
            log.info(f"Campaign {campaign}: too few treated/control households after exclusions, skipping direct estimate (needs pooling)")
            results.append({"CAMPAIGN": campaign, "N_TREATED": len(treated_rows), "N_CONTROL": len(control_rows), "DID_ESTIMATE_PER_WEEK": np.nan, "NOTE": "insufficient sample -- requires hierarchical pooling"})
            continue

        combo = pd.concat([treated_rows.assign(_T=1), control_rows.assign(_T=0)], ignore_index=True)
        feature_cols = demo_cols + ["Y_PRE_SALES"]
        X = pd.get_dummies(combo[feature_cols], dummy_na=True).fillna(0)

        try:
            ps_model = LogisticRegression(max_iter=1000)
            ps_model.fit(X, combo["_T"])
            combo["_pscore"] = ps_model.predict_proba(X)[:, 1]
        except Exception as e:
            log.warning(f"Campaign {campaign}: propensity model failed ({e}), using pre-period sales only for matching")
            combo["_pscore"] = combo["Y_PRE_SALES"]

        treated_idx = combo[combo["_T"] == 1].index
        control_idx = combo[combo["_T"] == 0].index
        nn = NearestNeighbors(n_neighbors=1).fit(combo.loc[control_idx, ["_pscore"]])
        _, match_pos = nn.kneighbors(combo.loc[treated_idx, ["_pscore"]])
        matched_control_idx = combo.loc[control_idx].iloc[match_pos.flatten()].index

        treated_during = combo.loc[treated_idx, outcome_col].to_numpy()
        treated_pre = combo.loc[treated_idx, "Y_PRE_SALES"].to_numpy()
        control_during = combo.loc[matched_control_idx, outcome_col].to_numpy()
        control_pre = combo.loc[matched_control_idx, "Y_PRE_SALES"].to_numpy()

        did_per_pair = (treated_during - treated_pre) - (control_during - control_pre)
        did_estimate = np.nanmean(did_per_pair) / max(duration_weeks, 1e-9)
        se = np.nanstd(did_per_pair, ddof=1) / np.sqrt(len(did_per_pair)) / max(duration_weeks, 1e-9)

        results.append({
            "CAMPAIGN": campaign,
            "N_TREATED": len(treated_rows),
            "N_CONTROL": len(control_rows),
            "DID_ESTIMATE_PER_WEEK": did_estimate,
            "SE_PER_WEEK": se,
            "NOTE": "matched, single campaign -- pool across campaigns before reporting",
        })

    return pd.DataFrame(results)


# --------------------------------------------------------------------------- #
# 6. Orchestration
# --------------------------------------------------------------------------- #

def run_pipeline(cfg: Config) -> dict:
    tables = load_reference_tables(cfg)
    episodes = build_episode_table(tables)
    audit_summary = audit_transactions(cfg)

    # Determine which households we need window-outcomes for: everyone who
    # appears in the transaction file, so control candidates are available too.
    # Cheap pass just to collect the household id universe before the full compute.
    households_in_scope = set()
    for chunk in pd.read_csv(cfg.paths["transaction"], chunksize=cfg.transaction_chunksize, usecols=["household_key"]):
        households_in_scope |= set(chunk["household_key"].unique())

    universal_outcomes = compute_household_campaign_outcomes(
        cfg,
        tables["campaign_desc"],
        dedupe_coupon_table(tables["coupon"]),
        tables["product"],
        households_in_scope,
    )

    # Attach the treated-side outcomes onto the episode table for reference/export.
    episodes_with_outcomes = episodes.merge(
        universal_outcomes, on=["HOUSEHOLD_KEY", "CAMPAIGN"], how="left"
    )
    outcome_cols = [c for c in universal_outcomes.columns if c.startswith("Y_")]
    episodes_with_outcomes[outcome_cols] = episodes_with_outcomes[outcome_cols].fillna(0.0)

    plan1_results = run_plan1_event_study_did(episodes, universal_outcomes, tables["hh_demographic"])

    return {
        "episodes": episodes_with_outcomes,
        "audit_summary": audit_summary,
        "plan1_results": plan1_results,
    }


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", type=str, required=True, help="Directory containing all 7 CSVs")
    parser.add_argument("--out-dir", type=str, default="./pipeline_output")
    args = parser.parse_args()

    cfg = Config(data_dir=Path(args.data_dir))
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    outputs = run_pipeline(cfg)
    outputs["episodes"].to_csv(out_dir / "episode_table_with_outcomes.csv", index=False)
    outputs["plan1_results"].to_csv(out_dir / "plan1_did_results.csv", index=False)
    log.info(f"Done. Outputs written to {out_dir}")

usage: ipykernel_launcher.py [-h] --data-dir DATA_DIR [--out-dir OUT_DIR]
ipykernel_launcher.py: error: the following arguments are required: --data-dir


SystemExit: 2

In [14]:
"""
Campaign causal analysis pipeline.

Implements, in order:
  Part 1 : estimand definition (see project-breakdown-part1.md) + episode table
  Step 3 : transaction unit / zero audit
  Outcome construction (Y columns) from the transaction file
  Plan 1 : matched stacked event-study Difference-in-Differences

Assumptions baked in from prior analysis of the files (see project-breakdown-part1.md):
  - Treatment D=1 is CAMPAIGN ASSIGNMENT (campaign_table.csv), never redemption.
  - coupon.csv must be joined on (CAMPAIGN, COUPON_UPC), and deduplicated on
    (CAMPAIGN, COUPON_UPC, PRODUCT_ID) before counting eligible products.
  - Households linked to overlapping campaigns are flagged, not silently pooled.
  - Effects are reported per campaign-week, not raw campaign totals, because
    campaign duration ranges 33-162 days.

This has NOT been run against real data yet -- it was written against the
documented schemas only. Run scripts/smoke_test in this file's __main__
block after pointing CONFIG at your actual files, and inspect intermediate
outputs before trusting anything downstream.
"""

from __future__ import annotations

import logging
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("campaign_pipeline")


# --------------------------------------------------------------------------- #
# Config
# --------------------------------------------------------------------------- #

@dataclass
class Config:
    data_dir: Path
    campaign_desc_file: str = "campaign_desc.csv"
    campaign_table_file: str = "campaign_table.csv"
    coupon_file: str = "coupon.csv"
    coupon_redempt_file: str = "coupon_redempt.csv"
    product_file: str = "product.csv"
    hh_demographic_file: str = "hh_demographic.csv"
    transaction_file: str = "transaction_data.csv"

    # Analysis parameters -- these are choices, not facts from the files.
    # State them explicitly so they're easy to challenge/change.
    pre_period_weeks: int = 4          # weeks before campaign start used as baseline
    post_period_weeks: int = 4         # weeks after campaign end used for payback check
    transaction_chunksize: int = 1_000_000  # for the ~6M row file

    paths: dict = field(init=False)

    def __post_init__(self):
        self.paths = {
            "campaign_desc": self.data_dir / self.campaign_desc_file,
            "campaign_table": self.data_dir / self.campaign_table_file,
            "coupon": self.data_dir / self.coupon_file,
            "coupon_redempt": self.data_dir / self.coupon_redempt_file,
            "product": self.data_dir / self.product_file,
            "hh_demographic": self.data_dir / self.hh_demographic_file,
            "transaction": self.data_dir / self.transaction_file,
        }


# --------------------------------------------------------------------------- #
# 1. Loading
# --------------------------------------------------------------------------- #

def load_reference_tables(cfg: Config) -> dict[str, pd.DataFrame]:
    """Load the six small reference CSVs (not the transaction file)."""
    log.info("Loading reference tables")

    campaign_desc = pd.read_csv(cfg.paths["campaign_desc"])
    campaign_table = pd.read_csv(cfg.paths["campaign_table"])
    coupon = pd.read_csv(cfg.paths["coupon"])
    coupon_redempt = pd.read_csv(cfg.paths["coupon_redempt"])
    product = pd.read_csv(cfg.paths["product"])
    hh_demographic = pd.read_csv(cfg.paths["hh_demographic"])

    # Normalize column names defensively (source files have been observed to
    # vary in case/whitespace across this dataset family).
    for df in (campaign_desc, campaign_table, coupon, coupon_redempt, product, hh_demographic):
        df.columns = [c.strip().upper() for c in df.columns]

    return {
        "campaign_desc": campaign_desc,
        "campaign_table": campaign_table,
        "coupon": coupon,
        "coupon_redempt": coupon_redempt,
        "product": product,
        "hh_demographic": hh_demographic,
    }


def dedupe_coupon_table(coupon: pd.DataFrame) -> pd.DataFrame:
    """
    coupon.csv is an eligibility table with 5,164 exact-duplicate rows (4.15%)
    across (CAMPAIGN, COUPON_UPC, PRODUCT_ID) triplets. Drop exact duplicates
    only -- do NOT collapse legitimate one-coupon-to-many-product mappings.
    """
    before = len(coupon)
    coupon_dedup = coupon.drop_duplicates(
        subset=["CAMPAIGN", "COUPON_UPC", "PRODUCT_ID"]
    ).copy()
    dropped = before - len(coupon_dedup)
    log.info(f"coupon.csv: dropped {dropped} exact-duplicate rows ({dropped/before:.2%})")
    return coupon_dedup


# --------------------------------------------------------------------------- #
# 2. Episode table (Part 1 deliverable)
# --------------------------------------------------------------------------- #

def build_episode_table(tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    Build the household x campaign episode table.
    D (treatment indicator) = row exists here = household assigned to campaign.
    Redemption is kept as a separate outcome-adjacent column, never as D.
    """
    log.info("Building episode table")

    campaign_table = tables["campaign_table"]
    campaign_desc = tables["campaign_desc"]
    coupon = dedupe_coupon_table(tables["coupon"])
    coupon_redempt = tables["coupon_redempt"]
    hh_demographic = tables["hh_demographic"]

    episodes = campaign_table.merge(
        campaign_desc[["CAMPAIGN", "DESCRIPTION", "START_DAY", "END_DAY"]],
        on="CAMPAIGN",
        how="left",
        validate="many_to_one",
    )
    episodes["DURATION_DAYS"] = episodes["END_DAY"] - episodes["START_DAY"] + 1

    # --- concurrent campaigns: count of *other* campaigns this household is
    # also linked to, whose windows overlap this campaign's window.
    episodes = _add_concurrent_campaign_count(episodes, campaign_desc)

    # --- eligible product count per campaign (post-dedup)
    eligible_counts = (
        coupon.groupby("CAMPAIGN")["PRODUCT_ID"].nunique().rename("ELIGIBLE_PRODUCT_COUNT")
    )
    episodes = episodes.merge(eligible_counts, on="CAMPAIGN", how="left")

    # --- redemption flag + first redemption day (household x campaign)
    redempt_agg = (
        coupon_redempt.groupby(["household_key".upper(), "CAMPAIGN"])["DAY"]
        .min()
        .rename("FIRST_REDEMPTION_DAY")
        .reset_index()
    )
    episodes = episodes.merge(redempt_agg, on=["HOUSEHOLD_KEY", "CAMPAIGN"], how="left")
    episodes["REDEEMED"] = episodes["FIRST_REDEMPTION_DAY"].notna().astype(int)

    # --- demographics (kept as explicit categories; missing stays missing)
    episodes = episodes.merge(hh_demographic, on="HOUSEHOLD_KEY", how="left")
    episodes["HAS_DEMOGRAPHICS"] = episodes["AGE_DESC"].notna().astype(int) if "AGE_DESC" in episodes else np.nan

    log.info(f"Episode table: {len(episodes)} rows, {episodes['HOUSEHOLD_KEY'].nunique()} households")
    return episodes


def _add_concurrent_campaign_count(episodes: pd.DataFrame, campaign_desc: pd.DataFrame) -> pd.DataFrame:
    """
    For every episode (household, campaign), count how many OTHER campaigns
    that same household is linked to whose [START_DAY, END_DAY] overlaps this
    campaign's window. O(n_households * campaigns_per_household^2) via groupby
    -- fine at 7,208 rows / max 17 campaigns per household.
    """
    windows = campaign_desc.set_index("CAMPAIGN")[["START_DAY", "END_DAY"]]

    def _count_for_group(group: pd.DataFrame) -> pd.Series:
        camps = group["CAMPAIGN"].tolist()
        counts = []
        for c in camps:
            s0, e0 = windows.loc[c, "START_DAY"], windows.loc[c, "END_DAY"]
            n_overlap = 0
            for other in camps:
                if other == c:
                    continue
                s1, e1 = windows.loc[other, "START_DAY"], windows.loc[other, "END_DAY"]
                if s0 <= e1 and s1 <= e0:
                    n_overlap += 1
            counts.append(n_overlap)
        return pd.Series(counts, index=group.index)

    episodes = episodes.copy()
    episodes["N_CONCURRENT_CAMPAIGNS"] = (
        episodes.groupby("HOUSEHOLD_KEY", group_keys=False).apply(_count_for_group)
    )
    return episodes


# --------------------------------------------------------------------------- #
# 3. Transaction audit (Step 3)
# --------------------------------------------------------------------------- #

def audit_transactions(cfg: Config) -> dict:
    """
    Streams the transaction file in chunks (it's ~6M rows) and reports:
      - rows/units carried by extreme-quantity outliers (possible fuel mixing)
      - household-week combinations with zero shopping trips
      - basic negative/zero-value integrity checks
    Does not modify the file; returns a summary dict to inform filtering
    decisions made explicitly later (never silently).
    """
    log.info("Auditing transaction file (chunked)")

    qty_values = []
    total_rows = 0
    total_qty = 0.0
    neg_sales = 0
    neg_qty = 0
    hh_week_pairs = set()
    all_households = set()
    all_weeks = set()

    for chunk in pd.read_csv(cfg.paths["transaction"], chunksize=cfg.transaction_chunksize):
        chunk.columns = [c.strip() for c in chunk.columns]
        total_rows += len(chunk)
        total_qty += chunk["QUANTITY"].sum()
        neg_sales += (chunk["SALES_VALUE"] < 0).sum()
        neg_qty += (chunk["QUANTITY"] < 0).sum()

        qty_values.append(chunk["QUANTITY"].to_numpy())

        pairs = set(zip(chunk["household_key"], chunk["WEEK_NO"]))
        hh_week_pairs |= pairs
        all_households |= set(chunk["household_key"].unique())
        all_weeks |= set(chunk["WEEK_NO"].unique())

    qty_all = np.concatenate(qty_values)
    q99 = np.quantile(qty_all, 0.99)
    outlier_mask = qty_all >= q99
    outlier_row_share = outlier_mask.mean()
    outlier_unit_share = qty_all[outlier_mask].sum() / qty_all.sum()

    n_possible_hh_weeks = len(all_households) * len(all_weeks)
    no_trip_share = 1 - (len(hh_week_pairs) / n_possible_hh_weeks) if n_possible_hh_weeks else np.nan

    summary = {
        "total_rows": total_rows,
        "total_quantity": float(total_qty),
        "negative_sales_rows": int(neg_sales),
        "negative_quantity_rows": int(neg_qty),
        "top_1pct_qty_row_share": float(outlier_row_share),
        "top_1pct_qty_unit_share": float(outlier_unit_share),
        "n_households": len(all_households),
        "n_weeks": len(all_weeks),
        "household_week_no_trip_share": float(no_trip_share),
    }
    log.info(f"Audit summary: {summary}")
    log.warning(
        "top_1pct_qty_unit_share above should be compared against the PDF's claimed "
        "98.7%-units-in-1.2%-of-rows fuel-mixing figure. If far lower, this dataset's "
        "fuel contamination may differ from the PDF's reference dataset -- do not assume "
        "it transfers."
    )
    return summary


# --------------------------------------------------------------------------- #
# 4. Outcome construction (Y columns for the episode table)
# --------------------------------------------------------------------------- #

def compute_household_campaign_outcomes(
    cfg: Config,
    campaign_desc: pd.DataFrame,
    coupon: pd.DataFrame,
    product: pd.DataFrame,
    households_in_scope: set,
) -> pd.DataFrame:
    """
    Streams the transaction file ONCE and computes, for EVERY household in
    households_in_scope x EVERY campaign (not just linked pairs), outcomes
    over that campaign's calendar window:
      Y_ELIGIBLE_SALES / Y_ELIGIBLE_UNITS : sales of THAT campaign's eligible products
      Y_CATEGORY_SALES                    : sales in the SAME COMMODITY as that
                                             campaign's eligible products (not all sales)
      Y_RIVAL_SALES                       : Y_CATEGORY_SALES minus Y_ELIGIBLE_SALES
                                             (same commodity, non-eligible products)
      Y_PRE_SALES / Y_POST_SALES          : household total sales in the pre/post windows

    Computing this for every household (linked or not) against every campaign
    is what makes Plan 1's control group comparable: a control household's
    "during" outcome is now measured over the SAME calendar window as the
    campaign it's being matched against, not over its own unrelated episode.

    30 campaigns x ~2,500 households is a small enough combination to hold in
    memory; only the streaming transaction read needs chunking.
    """
    log.info("Computing household x campaign window outcomes (universal table)")

    # Eligible products per campaign, and the commodities those products belong to.
    eligible = coupon[["CAMPAIGN", "PRODUCT_ID"]].drop_duplicates()
    product_commodity = product.set_index("PRODUCT_ID")["COMMODITY_DESC"]
    eligible = eligible.assign(COMMODITY_DESC=eligible["PRODUCT_ID"].map(product_commodity))

    eligible_products_by_campaign = eligible.groupby("CAMPAIGN")["PRODUCT_ID"].apply(set).to_dict()
    eligible_commodities_by_campaign = (
        eligible.groupby("CAMPAIGN")["COMMODITY_DESC"].apply(lambda s: set(s.dropna())).to_dict()
    )

    windows = campaign_desc.set_index("CAMPAIGN")[["START_DAY", "END_DAY"]]
    campaigns = windows.index.tolist()

    accum = {col: {} for col in [
        "Y_ELIGIBLE_SALES", "Y_ELIGIBLE_UNITS", "Y_CATEGORY_SALES",
        "Y_RIVAL_SALES", "Y_PRE_SALES", "Y_POST_SALES",
    ]}

    for chunk in pd.read_csv(cfg.paths["transaction"], chunksize=cfg.transaction_chunksize):
        chunk.columns = [c.strip() for c in chunk.columns]
        chunk = chunk[chunk["household_key"].isin(households_in_scope)]
        if chunk.empty:
            continue
        chunk["COMMODITY_DESC"] = chunk["PRODUCT_ID"].map(product_commodity)

        # Vectorized per-campaign (30 iterations), not per-household (thousands
        # of iterations) -- each iteration is a handful of pandas groupby-sums
        # over the whole chunk rather than one over a tiny per-household slice.
        for campaign in campaigns:
            start, end = windows.loc[campaign, "START_DAY"], windows.loc[campaign, "END_DAY"]
            pre_start = start - cfg.pre_period_weeks * 7
            post_end = end + cfg.post_period_weeks * 7

            elig_products = eligible_products_by_campaign.get(campaign, set())
            elig_commodities = eligible_commodities_by_campaign.get(campaign, set())

            during = chunk[(chunk["DAY"] >= start) & (chunk["DAY"] <= end)]
            pre = chunk[(chunk["DAY"] >= pre_start) & (chunk["DAY"] < start)]
            post = chunk[(chunk["DAY"] > end) & (chunk["DAY"] <= post_end)]

            if len(during):
                is_elig = during["PRODUCT_ID"].isin(elig_products)
                same_commodity = during["COMMODITY_DESC"].isin(elig_commodities)

                elig_sales_by_hh = during.loc[is_elig].groupby("household_key")["SALES_VALUE"].sum()
                elig_units_by_hh = during.loc[is_elig].groupby("household_key")["QUANTITY"].sum()
                category_sales_by_hh = during.loc[same_commodity].groupby("household_key")["SALES_VALUE"].sum()

                for hh, val in elig_sales_by_hh.items():
                    accum["Y_ELIGIBLE_SALES"][(hh, campaign)] = accum["Y_ELIGIBLE_SALES"].get((hh, campaign), 0) + val
                for hh, val in elig_units_by_hh.items():
                    accum["Y_ELIGIBLE_UNITS"][(hh, campaign)] = accum["Y_ELIGIBLE_UNITS"].get((hh, campaign), 0) + val
                for hh, val in category_sales_by_hh.items():
                    key = (hh, campaign)
                    accum["Y_CATEGORY_SALES"][key] = accum["Y_CATEGORY_SALES"].get(key, 0) + val
                    accum["Y_RIVAL_SALES"][key] = accum["Y_RIVAL_SALES"].get(key, 0) + val - elig_sales_by_hh.get(hh, 0)

            if len(pre):
                pre_sales_by_hh = pre.groupby("household_key")["SALES_VALUE"].sum()
                for hh, val in pre_sales_by_hh.items():
                    key = (hh, campaign)
                    accum["Y_PRE_SALES"][key] = accum["Y_PRE_SALES"].get(key, 0) + val
            if len(post):
                post_sales_by_hh = post.groupby("household_key")["SALES_VALUE"].sum()
                for hh, val in post_sales_by_hh.items():
                    key = (hh, campaign)
                    accum["Y_POST_SALES"][key] = accum["Y_POST_SALES"].get(key, 0) + val

    all_keys = set()
    for d in accum.values():
        all_keys |= set(d.keys())
    idx = pd.MultiIndex.from_tuples(sorted(all_keys), names=["HOUSEHOLD_KEY", "CAMPAIGN"])
    out = pd.DataFrame(index=idx)
    for col, d in accum.items():
        out[col] = pd.Series(d)
    out = out.fillna(0.0).reset_index()

    log.info(f"Universal outcome table: {len(out)} household x campaign rows for {len(households_in_scope)} households")
    return out


# --------------------------------------------------------------------------- #
# 5. Plan 1: matched stacked event-study DiD
# --------------------------------------------------------------------------- #

def run_plan1_event_study_did(
    episodes: pd.DataFrame,
    universal_outcomes: pd.DataFrame,
    demographics: pd.DataFrame,
    outcome_col: str = "Y_ELIGIBLE_SALES",
) -> pd.DataFrame:
    """
    For each campaign:
      1. Treated = households assigned to this campaign with N_CONCURRENT_CAMPAIGNS == 0
         (concurrently-treated households are excluded from Plan 1, not modeled --
         Plan 1 can't separate simultaneous treatments).
      2. Controls = every OTHER household in universal_outcomes that is NOT linked
         to this campaign in episodes, regardless of what campaigns they're linked
         to elsewhere -- their outcome is pulled from universal_outcomes for THIS
         campaign's calendar window, so treated and control are compared over the
         identical time period. (Controls linked to a campaign that overlaps this
         one's window are still excluded, since their own treatment would confound
         the comparison.)
      3. Match treated to controls on available demographics + pre-period sales
         (nearest-neighbor on a propensity score from logistic regression).
      4. DiD estimate = (during_treated - pre_treated) - (during_control - pre_control),
         normalized to per-week.
    Returns one row per campaign with the DiD estimate and a naive matched-pair SE
    (a placeholder -- the PDF wants bootstrap/Fieller-style intervals at the ROI
    stage, not this SE; this is enough to rank campaigns directionally, not to
    report as a final interval).
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.neighbors import NearestNeighbors

    log.info("Running Plan 1: matched stacked event-study DiD")
    results = []

    # household -> set of campaigns they're linked to, for exclusion checks
    linked_campaigns_by_hh = episodes.groupby("HOUSEHOLD_KEY")["CAMPAIGN"].apply(set).to_dict()
    concurrent_free_hh = set(episodes.loc[episodes["N_CONCURRENT_CAMPAIGNS"].eq(0), "HOUSEHOLD_KEY"])

    windows = episodes.drop_duplicates("CAMPAIGN").set_index("CAMPAIGN")[["START_DAY", "END_DAY", "DURATION_DAYS"]]
    all_campaign_windows = windows[["START_DAY", "END_DAY"]]

    demo_cols = [c for c in demographics.columns if c.endswith("_DESC")]
    uo = universal_outcomes.merge(demographics, on="HOUSEHOLD_KEY", how="left")

    for campaign in windows.index:
        c_start, c_end = windows.loc[campaign, ["START_DAY", "END_DAY"]]
        duration_weeks = windows.loc[campaign, "DURATION_DAYS"] / 7.0

        treated_hh = set(episodes.loc[episodes["CAMPAIGN"] == campaign, "HOUSEHOLD_KEY"]) & concurrent_free_hh

        def _overlaps_campaign(other_campaign: int) -> bool:
            o_start, o_end = all_campaign_windows.loc[other_campaign]
            return c_start <= o_end and o_start <= c_end

        overlapping_campaigns = {c for c in all_campaign_windows.index if c != campaign and _overlaps_campaign(c)}

        control_hh = {
            hh for hh, camps in linked_campaigns_by_hh.items()
            if hh not in treated_hh and not (camps & ({campaign} | overlapping_campaigns))
        }
        # Households present in the transaction data but never linked to any campaign are valid controls too.
        control_hh |= set(universal_outcomes["HOUSEHOLD_KEY"].unique()) - set(linked_campaigns_by_hh.keys()) - treated_hh

        treated_rows = uo[(uo["CAMPAIGN"] == campaign) & (uo["HOUSEHOLD_KEY"].isin(treated_hh))].copy()
        control_rows = uo[(uo["CAMPAIGN"] == campaign) & (uo["HOUSEHOLD_KEY"].isin(control_hh))].copy()

        if len(treated_rows) < 10 or len(control_rows) < 10:
            log.info(f"Campaign {campaign}: too few treated/control households after exclusions, skipping direct estimate (needs pooling)")
            results.append({"CAMPAIGN": campaign, "N_TREATED": len(treated_rows), "N_CONTROL": len(control_rows), "DID_ESTIMATE_PER_WEEK": np.nan, "NOTE": "insufficient sample -- requires hierarchical pooling"})
            continue

        combo = pd.concat([treated_rows.assign(_T=1), control_rows.assign(_T=0)], ignore_index=True)
        feature_cols = demo_cols + ["Y_PRE_SALES"]
        X = pd.get_dummies(combo[feature_cols], dummy_na=True).fillna(0)

        try:
            ps_model = LogisticRegression(max_iter=1000)
            ps_model.fit(X, combo["_T"])
            combo["_pscore"] = ps_model.predict_proba(X)[:, 1]
        except Exception as e:
            log.warning(f"Campaign {campaign}: propensity model failed ({e}), using pre-period sales only for matching")
            combo["_pscore"] = combo["Y_PRE_SALES"]

        treated_idx = combo[combo["_T"] == 1].index
        control_idx = combo[combo["_T"] == 0].index
        nn = NearestNeighbors(n_neighbors=1).fit(combo.loc[control_idx, ["_pscore"]])
        _, match_pos = nn.kneighbors(combo.loc[treated_idx, ["_pscore"]])
        matched_control_idx = combo.loc[control_idx].iloc[match_pos.flatten()].index

        treated_during = combo.loc[treated_idx, outcome_col].to_numpy()
        treated_pre = combo.loc[treated_idx, "Y_PRE_SALES"].to_numpy()
        control_during = combo.loc[matched_control_idx, outcome_col].to_numpy()
        control_pre = combo.loc[matched_control_idx, "Y_PRE_SALES"].to_numpy()

        did_per_pair = (treated_during - treated_pre) - (control_during - control_pre)
        did_estimate = np.nanmean(did_per_pair) / max(duration_weeks, 1e-9)
        se = np.nanstd(did_per_pair, ddof=1) / np.sqrt(len(did_per_pair)) / max(duration_weeks, 1e-9)

        results.append({
            "CAMPAIGN": campaign,
            "N_TREATED": len(treated_rows),
            "N_CONTROL": len(control_rows),
            "DID_ESTIMATE_PER_WEEK": did_estimate,
            "SE_PER_WEEK": se,
            "NOTE": "matched, single campaign -- pool across campaigns before reporting",
        })

    return pd.DataFrame(results)


# --------------------------------------------------------------------------- #
# 6. Orchestration
# --------------------------------------------------------------------------- #

def run_pipeline(cfg: Config) -> dict:
    tables = load_reference_tables(cfg)
    episodes = build_episode_table(tables)
    audit_summary = audit_transactions(cfg)

    # Determine which households we need window-outcomes for: everyone who
    # appears in the transaction file, so control candidates are available too.
    # Cheap pass just to collect the household id universe before the full compute.
    households_in_scope = set()
    for chunk in pd.read_csv(cfg.paths["transaction"], chunksize=cfg.transaction_chunksize, usecols=["household_key"]):
        households_in_scope |= set(chunk["household_key"].unique())

    universal_outcomes = compute_household_campaign_outcomes(
        cfg,
        tables["campaign_desc"],
        dedupe_coupon_table(tables["coupon"]),
        tables["product"],
        households_in_scope,
    )

    # Attach the treated-side outcomes onto the episode table for reference/export.
    episodes_with_outcomes = episodes.merge(
        universal_outcomes, on=["HOUSEHOLD_KEY", "CAMPAIGN"], how="left"
    )
    outcome_cols = [c for c in universal_outcomes.columns if c.startswith("Y_")]
    episodes_with_outcomes[outcome_cols] = episodes_with_outcomes[outcome_cols].fillna(0.0)

    plan1_results = run_plan1_event_study_did(episodes, universal_outcomes, tables["hh_demographic"])

    return {
        "episodes": episodes_with_outcomes,
        "audit_summary": audit_summary,
        "plan1_results": plan1_results,
    }


def main(data_dir: str, out_dir: str = "./pipeline_output"):
    """Callable directly from a notebook: main('/path/to/csvs', './pipeline_output')"""
    cfg = Config(data_dir=Path(data_dir))
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    outputs = run_pipeline(cfg)
    outputs["episodes"].to_csv(out_path / "episode_table_with_outcomes.csv", index=False)
    outputs["plan1_results"].to_csv(out_path / "plan1_did_results.csv", index=False)
    log.info(f"Done. Outputs written to {out_path}")
    return outputs


if __name__ == "__main__":
    import sys
    import argparse

    # Jupyter/IPython injects its own kernel-connection args into sys.argv,
    # which argparse chokes on. Detect that case and fall back to editable
    # variables below instead of forcing you to run this as a .py script.
    running_in_notebook = "ipykernel_launcher" in sys.argv[0] or "ipykernel" in sys.modules

    if running_in_notebook:
        log.info("Detected notebook environment -- skipping argparse. Edit DATA_DIR/OUT_DIR below and re-run this cell.")
        DATA_DIR = "data"   # <-- EDIT THIS
        OUT_DIR = "./pipeline_output"     # <-- EDIT THIS if you want a different output location
        outputs = main(DATA_DIR, OUT_DIR)
    else:
        parser = argparse.ArgumentParser()
        parser.add_argument("--data-dir", type=str, required=True, help="Directory containing all 7 CSVs")
        parser.add_argument("--out-dir", type=str, default="./pipeline_output")
        args = parser.parse_args()
        outputs = main(args.data_dir, args.out_dir)

2026-08-06 12:20:52,544 | INFO | Detected notebook environment -- skipping argparse. Edit DATA_DIR/OUT_DIR below and re-run this cell.
2026-08-06 12:20:52,545 | INFO | Loading reference tables
2026-08-06 12:20:52,618 | INFO | Building episode table
2026-08-06 12:20:52,625 | INFO | coupon.csv: dropped 5164 exact-duplicate rows (4.15%)
2026-08-06 12:20:53,520 | INFO | Episode table: 7208 rows, 1584 households
2026-08-06 12:20:53,520 | INFO | Auditing transaction file (chunked)
2026-08-06 12:20:54,819 | INFO | Audit summary: {'total_rows': 2595732, 'total_quantity': 260685622.0, 'negative_sales_rows': 0, 'negative_quantity_rows': 0, 'top_1pct_qty_row_share': 0.010642470023869952, 'top_1pct_qty_unit_share': 0.9873510054958076, 'n_households': 2500, 'n_weeks': 102, 'household_week_no_trip_share': 0.5138196078431372}
2026-08-06 12:20:54,820 | WARNING | top_1pct_qty_unit_share above should be compared against the PDF's claimed 98.7%-units-in-1.2%-of-rows fuel-mixing figure. If far lower, this